# 03 Modelo final para Power BI

Construir las tablas de la capa gold. Tres tablas de hechos y dos dimensiones, todas en Europa.
Modelo en estrella, relaciones de muchos a uno desde cada hecho a su dimensión.


In [16]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

RAIZ = Path.cwd().parent if Path.cwd().name == "01_notebooks" else Path.cwd()
SILVER = RAIZ / "02_datos" / "silver"
GOLD = RAIZ / "02_datos" / "gold"
GOLD.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
registro = []
def registrar(paso, que_se_hizo, filas):
    registro.append({"paso": paso, "que_se_hizo": que_se_hizo, "filas": filas})

## Carga de silver

In [17]:
ids = pd.read_csv(SILVER / "ids_decomisos_limpio.csv.gz", parse_dates=["fecha"])
anual71 = pd.read_csv(SILVER / "unodc_71_anual.csv")
paises71 = pd.read_csv(SILVER / "unodc_71_paises.csv")
contexto = pd.read_csv(SILVER / "contexto_largo.csv")
paises_ctx = pd.read_csv(SILVER / "contexto_paises.csv")

for nombre, t in [("IDS", ids), ("7.1", anual71), ("contexto", contexto)]:
    print(f"{nombre}: {len(t):,} filas")
registrar(1, "Carga de silver", len(ids) + len(anual71) + len(contexto))

IDS: 718,842 filas
7.1: 8,002 filas
contexto: 3,234 filas


## Países de Europa
Se queda con Europa según la clasificación de UNODC. IDS ya solo trae países europeos. Liechtenstein y Mónaco están en IDS pero no en 7.1, así que se añaden desde IDS.

Para nombre y región se usa primero 7.1, luego IDS y por último el CPI.

IDS escribe la subregión de Liechtenstein y Mónaco como West and Central Europe, y 7.1 como Western and Central Europe. Se unifica con el nombre de 7.1.

In [18]:
paises_ids = ids[["iso3", "pais", "region", "subregion"]].drop_duplicates("iso3")
base = (paises71.set_index("iso3")[["pais", "region", "subregion"]]
        .combine_first(paises_ids.set_index("iso3")[["pais", "region", "subregion"]])
        .combine_first(paises_ctx.set_index("iso3")[["pais"]]))
base = base[base["region"] == "Europe"].reset_index()

dim_pais = base.merge(paises_ctx[["iso3", "region_cpi"]], on="iso3", how="left")
dim_pais["subregion"] = dim_pais["subregion"].replace({"West and Central Europe": "Western and Central Europe"}).fillna("Sin clasificar")
dim_pais["en_ids"] = dim_pais["iso3"].isin(paises_ids["iso3"])
dim_pais["en_anual"] = dim_pais["iso3"].isin(paises71["iso3"])
dim_pais["en_cpi"] = dim_pais["iso3"].isin(contexto.query("indicador == 'CPI'")["iso3"])
print(f"{len(dim_pais)} países de Europa: en IDS {dim_pais.en_ids.sum()}, en 7.1 {dim_pais.en_anual.sum()}, con CPI {dim_pais.en_cpi.sum()}")
display(dim_pais.subregion.value_counts())

47 países de Europa: en IDS 43, en 7.1 45, con CPI 42


subregion
Western and Central Europe    33
South-Eastern Europe          10
Eastern Europe                 4
Name: count, dtype: int64

## Frecuencia de reporting en IDS
Cobertura de cada país: meses con al menos un decomiso notificado dividido entre los meses del periodo (enero 2011 a junio 2026, 186 meses).
Los umbrales son una propuesta propia.

In [19]:
UMBRALES = [(0.80, "Alta"), (0.50, "Media"), (0.20, "Baja"), (0.0, "Muy baja")]
MESES_PERIODO = len(pd.period_range(ids["anio_mes"].min(), ids["anio_mes"].max(), freq="M"))

def clase(cobertura):
    for minimo, nombre in UMBRALES:
        if cobertura >= minimo:
            return nombre

meses = ids.groupby("iso3")["anio_mes"].agg(ids_meses_con_datos="nunique", ids_primer_mes="min", ids_ultimo_mes="max")
meses["ids_cobertura"] = (meses["ids_meses_con_datos"] / MESES_PERIODO).round(4)
meses["ids_frecuencia"] = meses["ids_cobertura"].map(clase)
dim_pais = dim_pais.merge(meses.reset_index(), on="iso3", how="left")
dim_pais["ids_frecuencia"] = dim_pais["ids_frecuencia"].fillna("Sin datos IDS")
ORDEN_FRECUENCIA = {"Alta": 1, "Media": 2, "Baja": 3, "Muy baja": 4, "Sin datos IDS": 5}
dim_pais["ids_frecuencia_orden"] = dim_pais["ids_frecuencia"].map(ORDEN_FRECUENCIA)
assert dim_pais["ids_frecuencia_orden"].notna().all()
assert dim_pais["iso3"].is_unique

print("Meses del periodo:", MESES_PERIODO)
display(dim_pais["ids_frecuencia"].value_counts().rename("países"))
registrar(2, "dim_pais de Europa con frecuencia de reporting", len(dim_pais))

Meses del periodo: 186


ids_frecuencia
Alta             13
Media            13
Muy baja         10
Baja              7
Sin datos IDS     4
Name: países, dtype: int64

## Nombres en español
UNODC y Transparency International dan los países, regiones y subregiones en inglés. Para que todo el modelo esté en un solo idioma se traducen. El nombre original se conserva en pais_unodc.

In [20]:
PAIS_ES = {
    "ALB": "Albania", "AND": "Andorra", "AUT": "Austria", "BEL": "Bélgica", "BGR": "Bulgaria", "BIH": "Bosnia y Herzegovina",
    "BLR": "Bielorrusia", "CHE": "Suiza", "CYP": "Chipre", "CZE": "Chequia", "DEU": "Alemania", "DNK": "Dinamarca",
    "ESP": "España", "EST": "Estonia", "FIN": "Finlandia", "FRA": "Francia", "GBR": "Reino Unido", "GIB": "Gibraltar",
    "GRC": "Grecia", "HRV": "Croacia", "HUN": "Hungría", "IRL": "Irlanda", "ISL": "Islandia", "ITA": "Italia",
    "LIE": "Liechtenstein", "LTU": "Lituania", "LUX": "Luxemburgo", "LVA": "Letonia", "MCO": "Mónaco", "MDA": "Moldavia",
    "MKD": "Macedonia del Norte", "MLT": "Malta", "MNE": "Montenegro", "NLD": "Países Bajos", "NOR": "Noruega",
    "POL": "Polonia", "PRT": "Portugal", "ROU": "Rumanía", "RUS": "Rusia", "SRB": "Serbia", "SVK": "Eslovaquia",
    "SVN": "Eslovenia", "SWE": "Suecia", "TUR": "Türkiye", "UKR": "Ucrania", "VAT": "Santa Sede", "XXK": "Kosovo",
}
SUBREGION_ES = {"Western and Central Europe": "Europa occidental y central", "South-Eastern Europe": "Europa sudoriental",
                "Eastern Europe": "Europa oriental"}
REGION_CPI_ES = {"WE/EU": "Europa occidental y UE", "ECA": "Europa oriental y Asia central"}

faltan = set(dim_pais["iso3"]) - set(PAIS_ES)
assert not faltan, f"Países sin traducir: {faltan}"
dim_pais["pais_unodc"] = dim_pais["pais"]
dim_pais["pais"] = dim_pais["iso3"].map(PAIS_ES)
dim_pais["region"] = "Europa"
dim_pais["subregion"] = dim_pais["subregion"].map(SUBREGION_ES).fillna("Sin clasificar")
dim_pais["region_cpi"] = dim_pais["region_cpi"].map(REGION_CPI_ES)
assert dim_pais["pais"].is_unique
display(dim_pais[["iso3", "pais", "pais_unodc", "subregion"]].head(8))

,iso3,pais,pais_unodc,subregion
0,ALB,Albania,Albania,Europa sudoriental
1,AND,Andorra,Andorra,Europa occidental y central
2,AUT,Austria,Austria,Europa occidental y central
3,BEL,Bélgica,Belgium,Europa occidental y central
4,BGR,Bulgaria,Bulgaria,Europa sudoriental
5,BIH,Bosnia y Herzegovina,Bosnia and Herzegovina,Europa sudoriental
6,BLR,Bielorrusia,Belarus,Europa oriental
7,CHE,Suiza,Switzerland,Europa occidental y central


## Grupo de droga común
IDS y 7.1 clasifican las drogas de forma distinta. Para filtrar las dos tablas con los mismos nombres se usa la clasificación de 7.1, traducida. Los grupos de IDS sin equivalente directo quedan como Sin equivalente en 7.1.

In [21]:
GRUPO_COCA = "Coca leaf and bush (not cocaine)"
TRADUCCION_GRUPO = {
    "Cannabis-type drugs (excluding synthetic cannabinoids)": "Cannabis",
    "Cocaine-type": "Cocaína y derivados",
    "Opioids": "Opioides",
    "Amphetamine-type stimulants (excluding “ecstasy”)": "Estimulantes tipo anfetamina",
    "“Ecstasy”-type substances": "Éxtasis",
    "Sedatives and tranquillizers": "Sedantes y tranquilizantes",
    "Hallucinogens": "Alucinógenos",
    "NPS": "Nuevas sustancias psicoactivas",
    "Precursors": "Precursores",
    "Solvents and Inhalants": "Solventes e inhalantes",
    "Any other drugs/substances": "Otras sustancias",
    GRUPO_COCA: "Hoja y arbusto de coca (no es cocaína)",
}
SIN_EQUIVALENTE = "Sin equivalente en 7.1"

sin_traducir = set(anual71["grupo_droga"]) - set(TRADUCCION_GRUPO)
assert not sin_traducir, f"Grupos de 7.1 sin traducir: {sin_traducir}"
print(len(TRADUCCION_GRUPO), "grupos de 7.1 traducidos")

12 grupos de 7.1 traducidos


## fact_decomiso_anual
Es la tabla de 7.1 en Europa. Una fila por país, año y grupo de droga, en kg.

In [22]:
fact_decomiso_anual = anual71[anual71["iso3"].isin(dim_pais["iso3"])].copy()
fact_decomiso_anual["grupo_droga"] = fact_decomiso_anual["grupo_droga"].map(TRADUCCION_GRUPO)
fact_decomiso_anual["fecha"] = pd.to_datetime(fact_decomiso_anual["anio"].astype(str) + "-12-31")
fact_decomiso_anual = fact_decomiso_anual[["fecha", "anio", "iso3", "grupo_droga", "kg", "metodo", "kg_detalle", "kg_total_reportado", "n_filas_detalle"]]
fact_decomiso_anual = fact_decomiso_anual.sort_values(["iso3", "anio", "grupo_droga"]).reset_index(drop=True)
assert not fact_decomiso_anual.duplicated(["iso3", "anio", "grupo_droga"]).any()
print(f"{len(fact_decomiso_anual):,} filas, {fact_decomiso_anual.iso3.nunique()} países")
registrar(3, "fact_decomiso_anual (7.1, Europa)", len(fact_decomiso_anual))

3,070 filas, 45 países


## Marcas en dim_pais para el análisis
- en_panel_cocaina: el país tiene cocaína notificada en 7.1 todos los años de 2015 a 2024. Así se comparan siempre los mismos países.
- en_panel_cannabis: lo mismo para el cannabis, sin Türkiye. Es el grupo de control.
- grupo_geografico: Mar del Norte (Bélgica, Países Bajos y Alemania) o Resto de Europa.

In [23]:
coca_pais = (fact_decomiso_anual[(fact_decomiso_anual.grupo_droga == "Cocaína y derivados") & fact_decomiso_anual.anio.between(2015, 2024)]
             .groupby("iso3")["anio"].nunique())
panel_iso3 = coca_pais[coca_pais == 10].index
dim_pais["en_panel_cocaina"] = dim_pais["iso3"].isin(panel_iso3)
cann_pais = (fact_decomiso_anual[(fact_decomiso_anual.grupo_droga == "Cannabis") & fact_decomiso_anual.anio.between(2015, 2024)]
             .groupby("iso3")["anio"].nunique())
dim_pais["en_panel_cannabis"] = dim_pais["iso3"].isin(cann_pais[cann_pais == 10].index) & (dim_pais["iso3"] != "TUR")
print(f"Países en el panel de cannabis (control, sin Türkiye): {dim_pais.en_panel_cannabis.sum()}")
assert dim_pais.en_panel_cannabis.sum() == 29
dim_pais["grupo_geografico"] = np.where(dim_pais["iso3"].isin(["BEL", "NLD", "DEU"]), "Mar del Norte", "Resto de Europa")
print(f"Países en el panel de cocaína: {dim_pais.en_panel_cocaina.sum()}")
assert dim_pais.en_panel_cocaina.sum() == 30

Países en el panel de cannabis (control, sin Türkiye): 29
Países en el panel de cocaína: 30


## fact_decomiso_dia
Es la tabla de IDS. Solo se usan cantidades en masa (kg), porque litros, unidades y plantas no se pueden sumar con kg.

Se suman los decomisos del mismo día, país y grupo de droga. Los posibles duplicados no se borran: se cuentan en n_posibles_duplicados y sus kg en kg_posibles_duplicados, para poder restarlos si se decide.

In [24]:
masa = ids[ids["tipo_medida"] == "masa"].copy()
masa["grupo_droga_comun"] = masa["grupo_unodc_71"].map(TRADUCCION_GRUPO).fillna(SIN_EQUIVALENTE)
masa["kg_posibles_duplicados"] = masa["cantidad_kg"].where(masa["posible_duplicado"])

CLAVE_DIA = ["fecha", "iso3", "grupo_droga_comun"]
fact_decomiso_dia = (masa.groupby(CLAVE_DIA, as_index=False)
                     .agg(n_decomisos=("id_decomiso", "size"),
                          kg=("cantidad_kg", lambda s: s.sum(min_count=1)),
                          n_posibles_duplicados=("posible_duplicado", "sum"),
                          kg_posibles_duplicados=("kg_posibles_duplicados", lambda s: s.sum(min_count=1)))
                     .rename(columns={"grupo_droga_comun": "grupo_droga"}))
fact_decomiso_dia["kg"] = fact_decomiso_dia["kg"].round(9)
fact_decomiso_dia["kg_posibles_duplicados"] = fact_decomiso_dia["kg_posibles_duplicados"].round(9)

assert fact_decomiso_dia["n_decomisos"].sum() == len(masa)
assert abs(fact_decomiso_dia["kg"].sum() - masa["cantidad_kg"].sum()) < 1e-6 * masa["cantidad_kg"].sum()
assert not fact_decomiso_dia.duplicated(["fecha", "iso3", "grupo_droga"]).any()
print(f"{len(masa):,} decomisos en masa a {len(fact_decomiso_dia):,} filas")
print(f"Se dejan fuera {len(ids) - len(masa):,} decomisos medidos en unidades, plantas, volumen u otro")
registrar(4, "fact_decomiso_dia (IDS, solo masa)", len(fact_decomiso_dia))
display(fact_decomiso_dia.head())

624,785 decomisos en masa a 106,391 filas
Se dejan fuera 94,057 decomisos medidos en unidades, plantas, volumen u otro


,fecha,iso3,grupo_droga,n_decomisos,kg,n_posibles_duplicados,kg_posibles_duplicados
0,2011-01-01,CHE,Cocaína y derivados,1,0.47900,0,NaN
1,2011-01-01,DEU,Cocaína y derivados,1,0.90000,0,NaN
2,2011-01-01,ESP,Cannabis,4,3051.06000,0,NaN
3,2011-01-01,ESP,Cocaína y derivados,2,2.62300,0,NaN
4,2011-01-01,PRT,Cocaína y derivados,2,3.80348,0,NaN


## fact_contexto
Los tres indicadores de contexto (CPI, gasto en protección social y AROPE) para los países de Europa.

In [25]:
fact_contexto = contexto[contexto["iso3"].isin(dim_pais["iso3"])].copy()
fact_contexto["fecha"] = pd.to_datetime(fact_contexto["anio"].astype(str) + "-12-31")
fact_contexto = fact_contexto[["fecha", "anio", "iso3", "indicador", "valor"]].reset_index(drop=True)
assert not fact_contexto.duplicated(["iso3", "anio", "indicador"]).any()
display(fact_contexto.groupby("indicador").agg(filas=("valor", "size"), paises=("iso3", "nunique"), desde=("anio", "min"), hasta=("anio", "max")))
registrar(5, "fact_contexto (Europa)", len(fact_contexto))

,filas,paises,desde,hasta
indicador,,,,
AROPE,332,35,2016,2025
CPI,588,42,2012,2025
GASTO_PS,411,37,2013,2024


## dim_fecha
Un día por fila, del 1 de enero de la primera fecha al 31 de diciembre de la última, sin huecos.

In [26]:
primero = min(fact_decomiso_dia.fecha.min().year, fact_decomiso_anual.anio.min(), fact_contexto.anio.min())
ultimo = max(fact_decomiso_dia.fecha.max().year, fact_decomiso_anual.anio.max(), fact_contexto.anio.max())
dim_fecha = pd.DataFrame({"fecha": pd.date_range(f"{primero}-01-01", f"{ultimo}-12-31")})
dim_fecha["anio"] = dim_fecha.fecha.dt.year
dim_fecha["mes"] = dim_fecha.fecha.dt.month
dim_fecha["trimestre"] = dim_fecha.fecha.dt.quarter
dim_fecha["anio_mes"] = dim_fecha.fecha.dt.strftime("%Y-%m")
dim_fecha["dia_semana"] = dim_fecha.fecha.dt.dayofweek + 1
print(f"{len(dim_fecha):,} días, de {primero} a {ultimo}")
registrar(6, "dim_fecha", len(dim_fecha))

5,844 días, de 2011 a 2026


## Puertos de cocaína en IDS
Para el análisis de los puertos del Mar del Norte (notebook 04). Solo decomisos de cocaína hechos en puerto, por año, país y puerto.

En IDS cada país escribe la ciudad a su manera, así que se unifican los nombres con una lista de alias que se amplía a mano: Antwerpen y Antwerp son el mismo puerto, igual que las distintas formas de Ceuta, Las Palmas o Dublín.

In [27]:
ALIAS_PUERTO = {
    ("BEL", "Antwerpen"): "Antwerp",
    ("ESP", "Ceuta, Ciudad De"): "Ceuta",
    ("ESP", "Ceuta/ Ciudad De"): "Ceuta",
    ("ESP", "Palmas De Gran Canaria"): "Las Palmas de Gran Canaria",
    ("ESP", "Palmas De Gran Canaria (Las)"): "Las Palmas de Gran Canaria",
    ("IRL", "Dublin Port"): "Dublin",
}

puertos = masa[masa["es_puerto"] & (masa["grupo_droga_comun"] == "Cocaína y derivados")].copy()
puertos["puerto"] = [ALIAS_PUERTO.get((i, c), c) for i, c in zip(puertos["iso3"], puertos["ciudad"])]
puertos.loc[puertos["puerto"] == "Sin dato", "puerto"] = "Sin ciudad (" + puertos["iso3"] + ")"

puertos_cocaina_ids = (puertos.groupby(["anio", "iso3", "puerto"], as_index=False)
                       .agg(n_decomisos=("id_decomiso", "size"), kg=("cantidad_kg", "sum")))
puertos_cocaina_ids["kg"] = puertos_cocaina_ids["kg"].round(9)
assert abs(puertos_cocaina_ids["kg"].sum() - puertos["cantidad_kg"].sum()) < 1e-6 * puertos["cantidad_kg"].sum()

print(f"{len(puertos):,} decomisos a {len(puertos_cocaina_ids):,} filas, {puertos_cocaina_ids.groupby(['iso3', 'puerto']).ngroups} puertos")
display(puertos_cocaina_ids.groupby(["iso3", "puerto"]).kg.sum().sort_values(ascending=False).head(10).round(0))
registrar(7, "puertos_cocaina_ids", len(puertos_cocaina_ids))

2,225 decomisos a 456 filas, 194 puertos


iso3  puerto          
BEL   Antwerp             503472.0
ESP   Algeciras            54011.0
ITA   Gioia Tauro          51597.0
ESP   Valencia             48238.0
      Barcelona            38155.0
PRT   Setubal              15788.0
ESP   Sin ciudad (ESP)     13891.0
GRC   Piraeus              12376.0
DEU   Hamburg              10608.0
ESP   Vigo                 10129.0
Name: kg, dtype: float64

## Comprobaciones del modelo
Si falla alguna, el notebook se detiene. Así las relaciones de Power BI (todas de muchos a uno) funcionarán.

In [28]:
def contenido(hecho, col, dim, nombre):
    faltan = set(hecho[col].dropna()) - set(dim[col])
    assert not faltan, f"{nombre}: {col} sin dimensión, por ejemplo {sorted(faltan)[:5]}"

assert dim_pais["iso3"].is_unique and dim_fecha["fecha"].is_unique
for nombre, hecho in [("fact_decomiso_anual", fact_decomiso_anual), ("fact_decomiso_dia", fact_decomiso_dia), ("fact_contexto", fact_contexto)]:
    contenido(hecho, "iso3", dim_pais, nombre)
    contenido(hecho, "fecha", dim_fecha, nombre)
contenido(puertos_cocaina_ids, "iso3", dim_pais, "puertos_cocaina_ids")
print("Correcto: todas las claves de los hechos existen en su dimensión")
registrar(8, "Comprobaciones de claves", 0)

Correcto: todas las claves de los hechos existen en su dimensión


In [29]:
control = fact_decomiso_anual.query("iso3 == 'ESP' and anio == 2024 and grupo_droga == 'Cocaína y derivados'")
display(control[["anio", "iso3", "grupo_droga", "kg"]])
assert 120_000 < control["kg"].iloc[0] < 130_000

,anio,iso3,grupo_droga,kg
905,2024,ESP,Cocaína y derivados,124700.164


## Exportar
Fechas en formato AAAA-MM-DD. 

In [30]:
tablas = {"fact_decomiso_anual": fact_decomiso_anual, "fact_decomiso_dia": fact_decomiso_dia, "fact_contexto": fact_contexto,
          "dim_pais": dim_pais, "dim_fecha": dim_fecha, "puertos_cocaina_ids": puertos_cocaina_ids}
for nombre, tabla in tablas.items():
    tabla.to_csv(GOLD / f"{nombre}.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d", float_format="%.9f")
    print(f"{nombre:<24}{len(tabla):>9,} filas, {tabla.shape[1]} columnas")

log = pd.DataFrame(registro)
log.to_csv(SILVER / "log_modelo.csv", index=False, encoding="utf-8-sig")
display(log)

fact_decomiso_anual         3,070 filas, 9 columnas
fact_decomiso_dia         106,391 filas, 7 columnas
fact_contexto               1,331 filas, 5 columnas
dim_pais                       47 filas, 18 columnas
dim_fecha                   5,844 filas, 6 columnas
puertos_cocaina_ids           456 filas, 5 columnas


,paso,que_se_hizo,filas
0,1,Carga de silver,730078
1,2,dim_pais de Europa con frecuencia de reporting,47
2,3,"fact_decomiso_anual (7.1, Europa)",3070
3,4,"fact_decomiso_dia (IDS, solo masa)",106391
4,5,fact_contexto (Europa),1331
5,6,dim_fecha,5844
6,7,puertos_cocaina_ids,456
7,8,Comprobaciones de claves,0
